In [1]:
import torch
import torch.nn as nn

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split,Subset

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [3]:
dataset_path = "../data/raw"

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [5]:
base_dataset = datasets.ImageFolder(
    root=dataset_path,
)

print(len(base_dataset))

20638


In [6]:
train_size = int(0.8 * len(base_dataset))
test_size = len(base_dataset) - train_size

train_subset, test_subset = random_split(
    base_dataset,
    [train_size,test_size]
)

In [7]:
train_dataset = datasets.ImageFolder(
    root=dataset_path,
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    root=dataset_path,
    transform=test_transform
)

train_dataset = Subset(
    train_dataset,
    train_subset.indices
)

test_dataset = Subset(
    test_dataset,
    test_subset.indices
)

In [8]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [9]:
print(len(train_dataset))
print(len(test_dataset))

16510
4128


In [10]:
model = models.efficientnet_b0(
    weights="DEFAULT"
)

print(model)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [11]:
print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


In [12]:
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(
        in_features=1280,
        out_features=15
    )
)

In [13]:
model = model.to(device)

In [14]:
print(model.classifier)


Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=15, bias=True)
)


In [15]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(total_params)

4026763


In [16]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [17]:
epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    print(
        f"Epoch {epoch+1} Average Loss : {avg_loss}"
    )

Epoch 1 Average Loss : 0.29127030921991653
Epoch 2 Average Loss : 0.10347765740101919
Epoch 3 Average Loss : 0.08188324993703687
Epoch 4 Average Loss : 0.06366793749052597
Epoch 5 Average Loss : 0.057181345112193124
Epoch 6 Average Loss : 0.05384965725759741
Epoch 7 Average Loss : 0.05218962453770029
Epoch 8 Average Loss : 0.04621631186858768
Epoch 9 Average Loss : 0.043885312279740026
Epoch 10 Average Loss : 0.03818032021280127


In [18]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images,labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = torch.argmax(outputs,dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
accuracy = (correct / total) * 100
print("Test Accuracy :", accuracy)

Test Accuracy : 99.39437984496125


In [19]:
all_predictions = []
all_labels = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

In [20]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    all_labels,
    all_predictions
)
for i in range(len(cm)):
    accuracy = cm[i][i]/cm[i].sum()*100
    print("Class",i,"Accuracy :",accuracy,"%")

Class 0 Accuracy : 99.45945945945947 %
Class 1 Accuracy : 99.63898916967509 %
Class 2 Accuracy : 100.0 %
Class 3 Accuracy : 98.46938775510205 %
Class 4 Accuracy : 100.0 %
Class 5 Accuracy : 100.0 %
Class 6 Accuracy : 98.21428571428571 %
Class 7 Accuracy : 99.72375690607734 %
Class 8 Accuracy : 98.50746268656717 %
Class 9 Accuracy : 99.71988795518207 %
Class 10 Accuracy : 98.83381924198251 %
Class 11 Accuracy : 99.32659932659934 %
Class 12 Accuracy : 99.20382165605095 %
Class 13 Accuracy : 100.0 %
Class 14 Accuracy : 100.0 %


In [22]:
print(len(train_dataset))
print(len(test_dataset))

16510
4128


In [23]:
torch.save(
    model.state_dict(),
    "efficientnet_b0_99_39.pth"
)

In [24]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print((correct / total) * 100)

99.67898243488794


In [25]:
print(
    len(
        set(train_subset.indices)
        &
        set(test_subset.indices)
    )
)

0
